In [1]:
# Install make notebook's active kernel; restart the kernel after installation.
%pip install "langchain[mcp]>=1.4.0" langchain-groq python-dotenv rich

Note: you may need to restart the kernel to use updated packages.


```mermaid
graph LR
    A[CineBot Agent] -->|MCPAdapter| B[(MCP Server)]
    B --> C[check_showtimes]
    B --> D[cancel_booking]
    B --> E[get_seat_map]
```

In [2]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
assert os.environ.get("GROQ_API_KEY") is not None, "GROQ_API_KEY is not set in the environment variables."

from langchain.chat_models import init_chat_model

model = init_chat_model("groq:openai/gpt-oss-120b")

# MCP Adapter in LangChain

In [4]:
%%writefile cinebot_mcp_server.py
import argparse

from fastmcp import FastMCP
from fastmcp.server.auth import AccessToken, TokenVerifier
from mcp.types import ToolAnnotations

AUTH_TOKEN = "cinebot-secret-token"


class StaticTokenVerifier(TokenVerifier):
    """Accepts requests whose bearer token exactly matches AUTH_TOKEN."""

    async def verify_token(self, token: str) -> AccessToken | None:
        if token == AUTH_TOKEN:
            return AccessToken(token=token, client_id="cinebot-client", scopes=[])
        return None


mcp = FastMCP("CineBot", auth=StaticTokenVerifier())

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True,requiredAuthentication=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--http",
        action="store_true",
        help="Run the authenticated HTTP server instead of the default stdio server.",
    )
    args = parser.parse_args()

    if args.http:
        mcp.run(transport="http", port=8000)
    else:
        mcp.run(transport="stdio", show_banner=False)


Overwriting cinebot_mcp_server.py


In [5]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

C:\Users\pragg\AppData\Local\Temp\ipykernel_15620\2920300683.py:2: LangChainBetaWarning: `langchain.mcp` is in beta. It is actively being worked on, so the API may change.
  from langchain.mcp import MCPAdapter


In [7]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path(r"E:\6\Projects\Agentic AI 3.0\Class24\cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

Available tools: ['check_showtimes', 'cancel_booking', 'get_seat_map']


In [8]:
from langchain.agents import create_agent

# Call a tool without any AI or Agent


In [10]:
tool_call_result=''
async with MCPAdapter(cinebot_mcp_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well

    try:
        cinebot_agent = create_agent(model=model, tools=tools)
        result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        tool_call_result = result
        print("Agent invocation result:", result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result: [{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_9df4f3c1-903f-4452-9ad9-80ab2565c66c'}]
Agent invocation result: {'messages': [HumanMessage(content='What are the showtimes for Interstellar?', additional_kwargs={}, response_metadata={}, id='779a7ce9-4a53-4fed-9c89-65429916671f'), AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call check_showtimes with movie_title "Interstellar".', 'tool_calls': [{'id': 'fc_d3a1f6d1-c0a5-42e4-971b-4202d1bdf802', 'function': {'arguments': '{"movie_title":"Interstellar"}', 'name': 'check_showtimes'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 209, 'total_tokens': 254, 'completion_time': 0.094606784, 'completion_tokens_details': {'reasoning_tokens': 15}, 'prompt_time': 0.009322342, 'prompt_tokens_details': None, 'queue_time': 0.331806326, 'total_time': 0.103929126}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_be28f5c9f

In [11]:
from rich import print as rprint
rprint("[bold green]Direct tool invocation result:[/bold green]", tool_call_result)

Direct tool invocation result:
{
    'messages': [
        HumanMessage(
            content='What are the showtimes for Interstellar?',
            additional_kwargs={},
            response_metadata={},
            id='779a7ce9-4a53-4fed-9c89-65429916671f'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': 'We need to call check_showtimes with movie_title "Interstellar".',
                'tool_calls': [
                    {
                        'id': 'fc_d3a1f6d1-c0a5-42e4-971b-4202d1bdf802',
                        'function': {'arguments': '{"movie_title":"Interstellar"}', 'name': 'check_showtimes'},
                        'type': 'function'
                    }
                ]
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 45,
                    'prompt_tokens': 209,
                    'total_tokens': 254,
                    'completion_time': 0.094606784,
                    'completion_tokens_details': {'reasoning_tokens': 15},
                    'prompt_time': 0.009322342,
                    'prompt_tokens_details': None,
                    'queue_time': 0.331806326,
                    'total_time': 0.103929126
                },
                'model_name': 'openai/gpt-oss-120b',
                'system_fingerprint': 'fp_be28f5c9f2',
                'service_tier': 'on_demand',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'groq'
            },
            id='lc_run--01a133a4-2ece-7fa3-82f0-a7e2f2bb3898-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'fc_d3a1f6d1-c0a5-42e4-971b-4202d1bdf802',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 209,
                'output_tokens': 45,
                'total_tokens': 254,
                'output_token_details': {'reasoning': 15}
            }
        ),
        ToolMessage(
            content=[
                {'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_ddf8497f-3895-4da0-8bec-a8dbff9325fb'}
            ],
            name='check_showtimes',
            id='33bd80d9-cc32-489d-a4d5-29251cec2e39',
            tool_call_id='fc_d3a1f6d1-c0a5-42e4-971b-4202d1bdf802',
            artifact={'structured_content': {'result': '7:00 PM and 10:15 PM'}}
        ),
        AIMessage(
            content='Here are the current showtimes for **Interstellar**:\n\n- **7:00\u202fPM**\n- 
**10:15\u202fPM**\n\nLet me know if you’d like to see the seat map, make a reservation, or need any other 
details!',
            additional_kwargs={
                'reasoning_content': 'The user asks: "What are the showtimes for Interstellar?" We have just 
fetched showtimes: 7:00 PM and 10:15 PM. We need to respond with that info. Possibly ask if they want to book or 
need more info.'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 114,
                    'prompt_tokens': 249,
                    'total_tokens': 363,
                    'completion_time': 0.239793944,
                    'completion_tokens_details': {'reasoning_tokens': 54},
                    'prompt_time': 0.010379758,
                    'prompt_tokens_details': None,
                    'queue_time': 0.339706758,
                    'total_time': 0.250173702
                },
                'model_name': 'openai/gpt-oss-120b',
                'system_fingerprint': 'fp_068241849b',
                'service_tier': 'on_demand',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'groq'
            },
            i

In [12]:
tool_call_result['messages'][-1]

AIMessage(content='Here are the current showtimes for **Interstellar**:\n\n- **7:00\u202fPM**\n- **10:15\u202fPM**\n\nLet me know if you’d like to see the seat map, make a reservation, or need any other details!', additional_kwargs={'reasoning_content': 'The user asks: "What are the showtimes for Interstellar?" We have just fetched showtimes: 7:00 PM and 10:15 PM. We need to respond with that info. Possibly ask if they want to book or need more info.'}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 249, 'total_tokens': 363, 'completion_time': 0.239793944, 'completion_tokens_details': {'reasoning_tokens': 54}, 'prompt_time': 0.010379758, 'prompt_tokens_details': None, 'queue_time': 0.339706758, 'total_time': 0.250173702}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_068241849b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a133a4-34c5-7882-bf30-d315933435c8-0', tool_call

# 5. Tool Output: Multimodal Content Blocks

An MCP tool result isn't limited to text. `langchain.mcp` converts whatever the server sends —
text, images, embedded files — into standard LangChain **content blocks**, so a model sees a
uniform shape regardless of what kind of server produced it.

| MCP content type | Converts to |
|---|---|
| `TextContent` | text content block |
| `ImageContent` | image content block (base64 + mime type) |
| `ResourceLink` (image mime type) | image content block (by URL) |
| `ResourceLink` (other) | file content block |
| `EmbeddedResource` (text) | text content block |
| `EmbeddedResource` (blob) | image or file content block, by mime type |
| `AudioContent` | **not yet supported** — raises `NotImplementedError` |

# Structured Output 

In [13]:
async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call ={
        'name': 'get_seat_map',
        'args': {'movie_title': 'Interstellar'},
        'id': 'unique_call_id_12345',  # Optional: Provide a unique ID for the call
        'type':'tool_call'
    }

    message = await seat_map_tool.ainvoke(tool_call)
    print("Text Content (what my model reads)",message.content)
    print("Structured Data (what my code can use if required)",message.artifact)
    print(message.artifact['structured_content']['available_rows'])  # Accessing structured data directly

Text Content (what my model reads) [{'type': 'text', 'text': '{"movie":"Interstellar","available_rows":["A","B","C"],"sold_out_rows":["D"]}', 'id': 'lc_6d18745e-fbdf-4df0-89da-0e1bdf5632bd'}]
Structured Data (what my code can use if required) {'structured_content': {'movie': 'Interstellar', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}
['A', 'B', 'C']


# Error handling



| Failure | What happened | What the agent sees |
|---|---|---|
| Server tool **ran and reported failure** (`isError=True`) | The tool executed, then explicitly failed (bad input, business-rule violation) | A `ToolMessage` with `status="error"` — **the agent can read it and self-correct**, same as any other tool error in this course |
| **Transport / session failure** | The connection dropped, the server crashed, the process couldn't start | **Raises an exception** — there is no error `ToolMessage` for the model to reason about, because nothing came back to convert |


In [14]:
%%writefile cinebot_error_server.py
from fastmcp import FastMCP
mcp = FastMCP("CineBot")

@mcp.tool()
def risky_lookup(booking_id: str) -> str:
    """Look up a booking -- fails if the ID format is wrong."""
    if not booking_id.startswith("BK"):
        raise ValueError(f"Invalid booking ID format: {booking_id!r}. Expected it to start with 'BK'.")
    return f"Booking {booking_id}: confirmed."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_error_server.py


In [15]:
error_message_global = ''
error_transport = PythonStdioTransport(
    script_path=Path('cinebot_error_server.py'),
    log_file=Path('cinebot_error_server.log')
)

async with MCPAdapter(error_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    risky_lookup_tool = tools[0]

    bad_call = {'name': 'risky_lookup', 'args': {'booking_id': '12345'}, 'id': 'bad_call_001', 'type': 'tool_call'}

    error_message = await risky_lookup_tool.ainvoke(bad_call)
    print("Error message from tool:", error_message)
    print("Content:", error_message.content)
    error_message_global = error_message
    print(error_message)
  

Error message from tool: content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_e6bf5c2a-36a2-451e-b519-514f300fbabf'}] name='risky_lookup' tool_call_id='bad_call_001' status='error'
Content: [{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_e6bf5c2a-36a2-451e-b519-514f300fbabf'}]
content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_e6bf5c2a-36a2-451e-b519-514f300fbabf'}] name='risky_lookup' tool_call_id='bad_call_001' status='error'


In [16]:
error_message_global

ToolMessage(content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format: '12345'. Expected it to start with 'BK'.", 'id': 'lc_e6bf5c2a-36a2-451e-b519-514f300fbabf'}], name='risky_lookup', tool_call_id='bad_call_001', status='error')

In [17]:
from langchain.agents.middleware import HumanInTheLoopMiddleware



In [18]:
def is_destructive_tool_call(tool):
    # return message.type == "tool_call" and message.name == "cancel_booking"
    meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
    return bool(meta.get('destructive_hint'))


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    for t in tools:
        print(f"Tool: {t.name}, Destructive: {is_destructive_tool_call(t)}, Tools:{t}")


Tool: check_showtimes, Destructive: False, Tools:name='check_showtimes' description='Check available showtimes for a movie.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'movie_title': {'type': 'string'}}, 'required': ['movie_title']} metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.1.0'}}} handle_tool_error=<function _handle_mcp_tool_error at 0x0000020D1F9DB880> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool at 0x0000020D2131EDE0>
Tool: cancel_booking, Destructive: True, Tools:name='cancel_booking' description='Cancel an existing booking. Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, '_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.1.0'}}} 

In [19]:
# This cell contained pasted tool output. Re-run the tool-inspection cell above instead.


SyntaxError: invalid syntax (78780302.py, line 1)

In [ ]:
async with MCPAdapter(cinebot_transport) as adapter:
    tools = await adapter.list_tools()

    for t in tools:
        print(f"{t.name}: destructive={is_destructive_tool_call(t)}")

    # One HITL config, built dynamically from whatever the server marked destructive.
    interrupt_on = {
        t.name: {"allowed_decisions": ["approve", "edit", "reject"]}
        for t in tools
        if is_destructive_tool_call(t)
    }
    print("\ninterrupt_on config:", interrupt_on)

    hitl_middleware = HumanInTheLoopMiddleware(interrupt_on=interrupt_on)
    cinebot_guarded = create_agent(model=model, tools=tools, middleware=[hitl_middleware])

check_showtimes: destructive=False
cancel_booking: destructive=True
get_seat_map: destructive=False

interrupt_on config: {'cancel_booking': {'allowed_decisions': ['approve', 'edit', 'reject']}}


---
# 9. Elicitation: When the Server Needs to Ask a Question Mid-Call

Some MCP tools need clarification partway through a call — "confirm you want to proceed?".
`langchain.mcp` answers this **automatically with a LangGraph `interrupt()`** — every client
`MCPAdapter` builds is armed for it by default, no extra code needed to enable it.

```mermaid
sequenceDiagram
    participant Agent
    participant MCPAdapter
    participant Server
    Agent->>MCPAdapter: call cancel_booking(...)
    MCPAdapter->>Server: tools/call
    Server-->>MCPAdapter: needs more input (elicitation)
    MCPAdapter->>Agent: interrupt() -- pauses the graph
    Note over Agent: A human answers, outside the graph
    Agent->>MCPAdapter: Command(resume={"responses": {key: answer}})
    MCPAdapter->>Server: tools/call, retried with the answer
    Server-->>MCPAdapter: final result
```


In [ ]:
%%writefile cinebot_elicit_server.py
from fastmcp import FastMCP, Context

mcp = FastMCP("CineBotElicit")

@mcp.tool()
async def cancel_booking_confirm(booking_id: str, ctx: Context) -> str:
    """Cancel a booking, but ask the caller to confirm first."""
    result = await ctx.elicit(
        f"Confirm cancellation of booking {booking_id}?",
        response_type=bool,
    )
    if result.action != "accept":
        return f"Cancellation of {booking_id} aborted ({result.action})."
    if not result.data:
        return f"Cancellation of {booking_id} aborted (user said no)."
    return f"Booking {booking_id}: cancelled."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_elicit_server.py


In [ ]:
elicit_transport = PythonStdioTransport(
    script_path=Path("cinebot_elicit_server.py"),
    log_file=Path("cinebot_elicit_server.log"),
)

async with MCPAdapter(elicit_transport) as adapter:
    tools = await adapter.list_tools()
    confirm_tool = next(t for t in tools if t.name == "cancel_booking_confirm")

    tool_call = {
        "name": "cancel_booking_confirm",
        "args": {"booking_id": "BK123"},
        "id": "call_1",
        "type": "tool_call",
    }
    elicit_result = await confirm_tool.ainvoke(tool_call)
    print("status:", elicit_result.status)
    print("content:", elicit_result.content)


status: error
content: [{'type': 'text', 'text': 'elicitation via server-initiated requests is unavailable on 2026-07-28 connections.', 'id': 'lc_d6af8a0d-1a1e-4ed0-aea6-92a8edee0fd4'}]


# Connecting to Multiple Servers

# 10. Connecting to Multiple Servers

Real deployments rarely talk to just one MCP server. Two patterns, with a real tradeoff:

| Pattern | What it does | Tool naming | Protocol negotiation |
|---|---|---|---|
| **`MCPConfig` dict** | One aggregate connection across several servers | Prefixed by the config key you choose | **Shared** — the whole fleet negotiates down to the oldest protocol era any member requires |
| **`ClientGroup`** | Independent connections, one per server | Namespaced `{server}_{tool}` automatically | **Independent** — each member keeps its own protocol era and auth |

```mermaid
graph TB
    subgraph MCPConfig["MCPConfig -- one aggregate connection"]
        A1[Agent] --> C1[MCPAdapter]
        C1 --> S1[Server A]
        C1 --> S2[Server B]
        S1 -.shared era.-> S2
    end
    subgraph ClientGroup["ClientGroup -- independent connections"]
        A2[Agent] --> C2[MCPAdapter]
        C2 --> G[ClientGroup]
        G --> S3[Server A own era/auth]
        G --> S4[Server B own era/auth]
    end
```

In [ ]:
from fastmcp.mcp_config import MCPConfig

fleet_config: MCPConfig = {
    "mcpServers": {
        "cinebot": {
            "command": "python",
            "args": ["cinebot_mcp_server.py"],
        },
        "errors": {
            "command": "python",
            "args": ["cinebot_error_server.py"],
        },
    }
}

In [ ]:

async with MCPAdapter(fleet_config) as adapter:
    fleet_tools = await adapter.list_tools()
    print("MCPConfig fleet tools:", [t.name for t in fleet_tools])

[10/03/26 10:10:54] INFO     Proxy detected connected client - reusing existing session for all       proxy.py:1383
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

                    INFO     Proxy detected connected client - reusing existing session for all       proxy.py:1383
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

MCPConfig fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup']


In [ ]:
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup


In [ ]:
group = ClientGroup(
    {
        "cinebot": Client(Path("cinebot_mcp_server.py")),
        "errors": Client(Path("cinebot_error_server.py"),mode="legacy"),
        'time_track_server':Client('https://time-track-mcp-server.vercel.app/mcp/')
    }
)
async with MCPAdapter(group) as adapter:
    group_tools = await adapter.list_tools()
    print("ClientGroup fleet tools:", [t.name for t in group_tools])


ClientGroup fleet tools: ['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup', 'time_track_server_log_time', 'time_track_server_get_timesheet', 'time_track_server_get_project_summary', 'time_track_server_list_projects']


# Authentication

# 11. Authentication

MCP servers that aren't your own local script usually need auth. This is configured on the
**FastMCP client**, not on `MCPAdapter` itself — `MCPAdapter` just wraps whatever client you hand
it (or builds a default one for a bare URL).

| Need | Pattern |
|---|---|
| Bearer token | `Client(url, auth=token)` |
| OAuth 2.1 (discovery, browser redirect, token exchange) | `Client(url, auth="oauth")` |
| Persisted OAuth across runs | `Client(url, auth=OAuth(mcp_url=url, token_storage=...))` |
| Different auth per server | `ClientGroup` with a different `auth=` on each member `Client` |
| Per-user auth in a deployment | A custom auth handler resolves the caller server-side; the graph factory mints/exchanges a token per user |

```mermaid
sequenceDiagram
    participant App
    participant Client as fastmcp.Client
    participant Server
    App->>Client: Client(url, auth=token_or_"oauth")
    alt Bearer token
        Client->>Server: Authorization: Bearer <token>
        Server-->>Client: 200 OK
    else OAuth 2.1
        Client->>Server: discovery + dynamic client registration
        Server-->>Client: browser redirect
        Client->>Server: token exchange
        Server-->>Client: access token
    end
```

In [ ]:
async def load_tools_with_bearer(url: str, token: str):
    # `auth` accepts a bearer-token string, the literal "oauth", or any httpx.Auth.
    async with MCPAdapter(Client(url, auth=token)) as adapter:
        return await adapter.list_tools()

In [ ]:
# Must match AUTH_TOKEN in cinebot_mcp_server.py
CINEBOT_AUTH_TOKEN = "cinebot-secret-token"

# Run the authenticated HTTP server in a separate terminal with:
#   python cinebot_mcp_server.py --http
cinebot_tools = await load_tools_with_bearer("http://127.0.0.1:8000/mcp", CINEBOT_AUTH_TOKEN)
print("Tools over HTTP with bearer auth:", [t.name for t in cinebot_tools])

# 12. `langchain.mcp` vs. `langchain-mcp-adapters` — What Actually Changed

If you (or a client) already has an MCP integration built on the older, separate
`langchain-mcp-adapters` package, here's the honest diff — including what the new API **doesn't**
do yet.

| | `langchain-mcp-adapters` (older, separate package) | `langchain.mcp` (this notebook, beta, built-in) |
|---|---|---|
| Install | `pip install langchain-mcp-adapters` | `pip install "langchain[mcp]>=1.4.0"` — no separate package |
| Entry point | `MultiServerMCPClient({...}).get_tools()` | `async with MCPAdapter(target) as adapter: await adapter.list_tools()` |
| Resources (`get_resources`) | ✅ Supported | ❌ **Not exposed by `MCPAdapter`** — drop to the underlying `fastmcp.Client` directly if you need this |
| Prompts (`get_prompt`) | ✅ Supported | ❌ **Not exposed by `MCPAdapter`** — same workaround |
| Tool call interceptors (`tool_interceptors`, logging/retry/`Command`-update patterns) | ✅ Supported, via `MCPToolCallRequest` | ❌ **No interceptor hook in this beta** — customize by wrapping tools yourself after `list_tools()` |
| Progress callbacks | ✅ Supported | Not part of the public `MCPAdapter` surface reviewed here |
| Structured content | Via `ToolMessage` content | `ToolMessage.artifact["structured_content"]` (typed as `MCPToolArtifact`) |
| Error handling | `handle_tool_errors` boolean flag | `isError=True` → `status="error"` automatically; transport errors raise either way |
| Tool metadata | Ungrouped | Single `tool.metadata["mcp"]` namespace (`tool.annotations`, `_meta`, `server`) |
| Elicitation | Not built in | Automatic, via LangGraph `interrupt()` |
| Status | Mature, stable | **Beta** — "actively being worked on, so the API may change" (the module's own words) |

> 🎤 **Teaching Note (for you):** the honest takeaway for a client engagement — if you need
> resources, prompts, or interceptor-style middleware around MCP calls TODAY, `langchain-mcp-adapters`
> still does things `langchain.mcp` doesn't yet. If you're starting fresh and only need tools, the
> new built-in path is where LangChain is clearly headed, and it's what today's official docs lead
> with. Don't present this as a strict "old bad, new good" — it's "different surface area, and the
> new one isn't a full superset yet."